In [ ]:
from tree_sitter import Parser
import tree_sitter_language_pack
import re
import spacy

# Load spaCy model for NER (ensure 'en_core_web_sm' is installed)
# nlp = spacy.load("en_core_web_sm")

def extract_macros(node, source_code: bytes, macros: dict, context: list):
    """
    Extract macros into `macros` dict, recording the active preprocessor context.
    """
    name_node = node.child_by_field_name('name')
    if not name_node:
        return

    identifier = source_code[name_node.start_byte:name_node.end_byte].decode()
    content    = source_code[node.start_byte:node.end_byte].decode()
    entry = {
        'content': content,
        'context': context.copy()
    }

    if node.type == 'preproc_def':
        value_node = node.child_by_field_name('value')
        entry['value'] = (
            source_code[value_node.start_byte:value_node.end_byte].decode()
            if value_node else ''
        )

    elif node.type == 'preproc_function_def':
        value_node  = node.child_by_field_name('value')
        params_node = node.child_by_field_name('parameters')
        entry['value']      = (
            source_code[value_node.start_byte:value_node.end_byte].decode()
            if value_node else ''
        )
        entry['parameters'] = (
            source_code[params_node.start_byte:params_node.end_byte].decode()
            if params_node else ''
        )

    macros[identifier] = entry

def extract_comment(node, source_code: bytes) -> str:
    """
    Clean and mask a block comment; return None for inline, copyright, or single-line comments.
    """
    comment_text = source_code[node.start_byte:node.end_byte].decode()
    text = comment_text.strip()
    # ignore single-line comments entirely
    if '\n' not in text:
        return None
    # ignore inline-style
    if text.startswith("//"):
        return None
    # ignore copyright/license
    low = text.lower()
    if any(word in low for word in ["copyright", "license"]):
        return None
    # strip comment markers
    if text.startswith("/*"): text = text[2:]
    if text.endswith("*/"): text = text[:-2]
    # remove leading/trailing non-alphanumeric
    lines = text.splitlines()
    cleaned = [re.sub(r'^[^A-Za-z0-9]+|[^A-Za-z0-9]+$', '', line).strip() for line in lines]
    combined = ' '.join(cleaned)
    # mask emails
    combined = re.sub(r'\b[\w\.-]+@[\w\.-]+\.(?:[A-Za-z]{2,})\b', '<EMAIL>', combined)
    return combined or None


def extract_function(node, source_code: bytes, context: list) -> dict:
    """
    Given a function_definition node and its context, extract its signature, body, and context.
    """
    identifier = None
    return_type = None
    type_qualifier = None
    storage_class = None
    params = []

    # slice full text
    func_content = source_code[node.start_byte:node.end_byte].decode()

    for child in node.children:
        text = source_code[child.start_byte:child.end_byte].decode()
        if child.type == 'type_qualifier':
            type_qualifier = text
        elif child.type == 'storage_class_specifier':
            storage_class = text
        elif child.type in ('type_identifier', 'primitive_type'):
            return_type = text
        elif child.type == 'pointer_declarator':
            if return_type:
                return_type += ' pointer'
            for pd_child in child.children:
                if pd_child.type == 'function_declarator':
                    for _fd in pd_child.children:
                        if _fd.type == 'identifier':
                            identifier = source_code[_fd.start_byte:_fd.end_byte].decode()
                        elif _fd.type == 'parameter_list':
                            for _p in _fd.children:
                                if _p.type == 'parameter_declaration':
                                    params.append(source_code[_p.start_byte:_p.end_byte].decode().strip())
        elif child.type == 'function_declarator':
            for _fd in child.children:
                if _fd.type == 'identifier':
                    identifier = source_code[_fd.start_byte:_fd.end_byte].decode()
                elif _fd.type == 'parameter_list':
                    for _p in _fd.children:
                        if _p.type == 'parameter_declaration':
                            params.append(source_code[_p.start_byte:_p.end_byte].decode().strip())

    start_line = node.start_point[0] + 1
    end_line   = node.end_point[0]   + 1

    result = {
        'name':         identifier or '<anonymous>',
        'return_type':  return_type or '',
        'parameters':   params,
        'type':         type_qualifier,
        'storage':      storage_class,
        'start_line':   start_line,
        'end_line':     end_line,
        'body':         func_content,
        'context':      context.copy(),
    }
    return result


def extract_code_details(source_code: bytes, is_cpp: bool = False):
    """
    ...same docstring...
    Returns: (details_list, macros_dict)
    """
    parser = tree_sitter_language_pack.get_parser('cpp' if is_cpp else 'c')
    tree   = parser.parse(source_code)
    root   = tree.root_node

    details = []
    macros  = {}

    def process_children(parent, context=None):
        if context is None:
            context = []
        siblings = parent.named_children
        i = 0

        while i < len(siblings):
            node = siblings[i]

            # 1) collect macros with context
            if node.type in ('preproc_def', 'preproc_function_def'):
                extract_macros(node, source_code, macros, context)
                # descend in case nested defs
                process_children(node, context)
                i += 1
                continue

            # 2) update context on #ifdef/#elif/#else
            if node.type in ('preproc_ifdef', 'preproc_elif', 'preproc_else'):
                full = source_code[node.start_byte:node.end_byte].decode()
                first = full.splitlines()[0].strip()
                new_ctx = context + [first]
                process_children(node, new_ctx)
                i += 1
                continue

            # 3) comments + function logic (unchanged)…
            if node.type == 'comment':
                next_node = siblings[i+1] if i+1 < len(siblings) else None
                comment_text = extract_comment(node, source_code)
                if next_node and next_node.type == 'function_definition':
                    func = extract_function(next_node, source_code, context)
                    if comment_text:
                        func['comment'] = comment_text
                    details.append(func)
                    i += 2
                    continue
                else:
                    if comment_text:
                        details.append(comment_text)
                    i += 1
                    continue

            # 4) standalone function
            if node.type == 'function_definition':
                func = extract_function(node, source_code, context)
                details.append(func)
                i += 1
                continue

            # 5) recurse
            process_children(node, context)
            i += 1

    process_children(root, context=[])
    return details, macros

In [ ]:
with open('code.c', 'rb') as cf:
    code = cf.read()

In [ ]:
details, macros = extract_code_details(code)

In [ ]:
macros

In [ ]:
def print_tree(node, source_code: bytes, indent: int = 0):
    """
    Recursively print each node's type and text, indented by depth.
    """
    # extract the node text and strip newlines for readability
    text = source_code[node.start_byte:node.end_byte].decode('utf8', errors='replace').strip().replace('\n', '\\n')
    print('  ' * indent + f"{node} ", text)

    for child in node.children:
        print_tree(child, source_code, indent + 1)

parser = tree_sitter_language_pack.get_parser('c')
tree = parser.parse(code)
root = tree.root_node
print_tree(tree.root_node, code)

In [ ]:
root.children[-4].text

In [ ]:
from bm25s import is_list_of_list_of_type, _is_tuple_of_list_of_tokens

In [ ]:
ids, vocab = query_tokens

In [ ]:
query_tokens = tokenization.Tokenized(ids=ids, vocab=vocab)

In [ ]:
query_tokens = tokenization.convert_tokenized_to_string_list(query_tokens)

In [ ]:
 corpus = retriever.corpus

In [ ]:
corpus

In [ ]:
retriever.corpus

In [ ]:
tqdm_kwargs = {
    "total": len(query_tokens),
    "desc": "BM25S Retrieve",
    "leave": True,
    "disable": False,
}
topk_fn = partial(
    retriever._get_top_k_results,
    k=2,
    sorted=sorted,
    backend="numba",
    weight_mask=None,
)

In [ ]:
out = tqdm(map(topk_fn, query_tokens), **tqdm_kwargs)

In [ ]:
scores, indices = zip(*out)

In [ ]:
scores

In [ ]:
[retriever.vocab_dict[token] for token in query_tokens[0] if token in retriever.vocab_dict]

In [ ]:
retriever.vocab_dict

In [ ]:
query_tokens

In [ ]:
from rapidfuzz import process, fuzz

In [ ]:
process.extractOne('doe', retriever.vocab_dict.keys(), scorer=fuzz.WRatio, score_cutoff=50)